# ITO5202 Assessment 2: Machine Learning and Streaming Analytics

**Student ID:** 27602966  
**Dataset:** NYC TLC Yellow Taxi Trip Records, January–December 2023, and Taxi Zone Lookup  
**GitHub repository:** [ITO5202_Ass2_27602966](https://github.com/hncha9-eng/ITO5202_Ass2_27602966)

## Assessment Overview

This assessment extends the historical batch analysis completed in Assessment 1. The approved dataset is reused to train and evaluate a Spark ML model, then apply the persisted model to previously unseen records replayed through Kafka.

| Notebook section | Requirements addressed |
|---|---|
| Execution Environment | Runtime configuration, dependency versions and environment verification |
| Before Part A: Dataset Reuse and Train/Stream Split | Source validation, data preparation, reproducible partitioning and streaming subset persistence |
| Part A: The ML Pipeline (45%) | Task selection, feature engineering, model comparison, evaluation, model persistence and reload verification |
| Part B: Streaming, Kafka and Reflection (45%) | Held-out replay, Kafka producer, Structured Streaming consumer, windowed aggregation and embedded reflection |
| Code Quality and Notebook Documentation (5%) | Clear code, documented decisions and visible execution evidence throughout |
| GitHub and Documentation (5%) | Development history, required repository artefacts and reproducible README instructions |

The streaming subset will remain excluded from model fitting, tuning and selection. Part B will use the persisted feature engineering and prediction pipeline from Part A.

## Development and Commit Plan

Development is organised into 13 commit milestones. Commits are made as the corresponding work is completed and verified. Additional debugging or refinement commits may be included where needed.

The existing environment and taxi lookup commits establish the project foundation. The four required assessment phases are represented explicitly below.

| No. | Phase | Development milestone | Commit message |
|---|---|---|---|
| 1 | Foundation | Environment setup and initial reproduction documentation | Set up Spark and Kafka environment with reproducible documentation |
| 2 | Foundation | Add the taxi zone reference data | Add NYC taxi zone lookup data |
| 3 | Foundation | Organise the notebook around assessment requirements | Align assessment notebook with assessment requirements |
| 4 | Phase 1 | Standardise monthly schemas and document data quality checks | Standardise taxi schemas and validate data quality |
| 5 | Phase 1 | Implement the seeded train/stream split and verify saved streaming data | Implement reproducible split and persist streaming holdout |
| 6 | Phase 1 | Explore training data, justify features and construct the preprocessing pipeline | Explore training data and implement feature engineering |
| 7 | Phase 2 | Train competing models and present their evaluation results | Train and compare competing Spark ML models |
| 8 | Phase 2 | Justify the final model, persist its complete pipeline and verify reloaded predictions | Select and persist the verified best pipeline |
| 9 | Phase 3 | Implement the standalone producer with parameterised batches and publication timestamps | Implement held-out Kafka batch producer |
| 10 | Phase 3 | Test producer execution, retain batch logs and document exact Kafka commands | Verify producer timing and document Kafka execution |
| 11 | Phase 4 | Implement streaming ingestion, explicit JSON parsing and persisted-model inference with prediction sinks | Integrate Kafka streaming with persisted model inference |
| 12 | Phase 4 | Implement sliding windows, watermarking and visible aggregation results | Add watermarked sliding-window prediction monitoring |
| 13 | Phase 4 and submission | Complete the evidence-based reflection, reproduction documentation and notebook/PDF checks | Complete reflection and verify submission documentation |

### Required Phase Coverage

- **Phase 1 — Commits 4–6:** train/stream split, data exploration and feature engineering.
- **Phase 2 — Commits 7–8:** model training, comparison, evaluation and persistence.
- **Phase 3 — Commits 9–10:** Kafka producer development and local testing.
- **Phase 4 — Commits 11–13:** Structured Streaming consumer, windowed aggregation and reflection.

Commit identifiers will be recorded after the commits exist. Notebook outputs and written interpretations will provide evidence for the completed milestones.

## 1. Initialise Spark
Run this notebook inside the existing PySpark container, from the repository root mounted at `/home/student/work`. Save and shut down the setup notebook's kernel before starting this notebook, so the assessment uses the available memory and Spark UI port.

The verified environment uses Spark 3.5.5, Scala 2.12, two local worker threads, 2 GB driver memory, and the `America/New_York` session timezone. The Kafka connector is pinned to Spark 3.5.5 and Scala 2.12. Package arguments must be set before creating Spark in a fresh kernel. Python packages have already been installed during environment setup.


In [1]:
import os
import sys
from pathlib import Path

import pyspark

EXPECTED_SPARK_VERSION = "3.5.5"
if pyspark.__version__ != EXPECTED_SPARK_VERSION:
    raise RuntimeError(
        f"Expected PySpark {EXPECTED_SPARK_VERSION}, "
        f"found {pyspark.__version__}. Use the assessment's verified container."
    )

KAFKA_CONNECTOR = (
    "org.apache.spark:"
    "spark-sql-kafka-0-10_2.12:3.5.5"
)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f"--driver-memory 2g --packages {KAFKA_CONNECTOR} pyspark-shell"
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITO5202 Assessment 2")
    .master("local[2]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


### Verify the runtime
The output should match the setup notebook: Spark 3.5.5, Scala 2.12.18, `local[2]`, driver memory `2g`, and `America/New_York`. The small range below checks that a Spark action completes.


In [2]:
print("Python:", sys.version.split()[0])
print("Spark:", spark.version)
print("Scala:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("Timezone:", spark.conf.get("spark.sql.session.timeZone"))
print("Kafka connector:", KAFKA_CONNECTOR)
spark.range(3).show()


Python: 3.10.19
Spark: 3.5.5
Scala: 2.12.18
Master: local[2]
Driver memory: 2g
Timezone: America/New_York
Kafka connector: org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



## 2. Verify the reused dataset
All source paths are relative to the repository root. The expected inputs are the twelve original 2023 monthly Parquet files and `taxi_zone_lookup.csv` under `data/`. This check verifies file availability without scanning or counting the full dataset.

The large Parquet files remain outside Git tracking; `data/README.md` documents their download locations. The small taxi zone lookup can be included in the repository.


In [3]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
MONTHLY_FILES = [
    DATA_DIR / f"yellow_tripdata_2023-{month:02d}.parquet"
    for month in range(1, 13)
]
ZONE_FILE = DATA_DIR / "taxi_zone_lookup.csv"
required_files = MONTHLY_FILES + [ZONE_FILE]

print("Repository root:", PROJECT_ROOT)
missing_files = []
for source_file in required_files:
    exists = source_file.is_file()
    print(f"{'FOUND' if exists else 'MISSING'}: {source_file.name}")
    if not exists:
        missing_files.append(source_file.name)

if missing_files:
    raise FileNotFoundError(
        "Place the original source files in the repository's data folder. "
        f"Missing: {', '.join(missing_files)}"
    )


Repository root: /home/student/work
FOUND: yellow_tripdata_2023-01.parquet
FOUND: yellow_tripdata_2023-02.parquet
FOUND: yellow_tripdata_2023-03.parquet
FOUND: yellow_tripdata_2023-04.parquet
FOUND: yellow_tripdata_2023-05.parquet
FOUND: yellow_tripdata_2023-06.parquet
FOUND: yellow_tripdata_2023-07.parquet
FOUND: yellow_tripdata_2023-08.parquet
FOUND: yellow_tripdata_2023-09.parquet
FOUND: yellow_tripdata_2023-10.parquet
FOUND: yellow_tripdata_2023-11.parquet
FOUND: yellow_tripdata_2023-12.parquet
FOUND: taxi_zone_lookup.csv


## 3. Inspect the monthly schemas
Assessment 1 identified differences between January and later files, including the types of `passenger_count`, `RatecodeID`, and location IDs, and the capitalisation of the airport fee column. Inspect each source separately before combining months so incompatible Parquet types are visible.

The following table reports source metadata only. It does not fit preprocessing, select records for modelling, or inspect the held-out streaming subset. The fixed-seed training/streaming split will be implemented after consistent source loading and deterministic eligibility rules are defined. Learned transformations will be fitted using the training pool only.


In [4]:
monthly_schemas = {}
schema_rows = []

for source_file in MONTHLY_FILES:
    source_df = spark.read.parquet(str(source_file))
    source_schema = source_df.schema
    monthly_schemas[source_file.name] = source_schema
    column_types = {
        field.name: field.dataType.simpleString()
        for field in source_schema.fields
    }
    airport_columns = [
        field.name for field in source_schema.fields
        if field.name.lower() == "airport_fee"
    ]
    schema_rows.append((
        source_file.stem[-7:],
        column_types.get("passenger_count", "missing"),
        column_types.get("RatecodeID", "missing"),
        column_types.get("PULocationID", "missing"),
        column_types.get("DOLocationID", "missing"),
        ", ".join(airport_columns) or "missing",
    ))

schema_summary = spark.createDataFrame(
    schema_rows,
    schema=(
        "month string, passenger_count string, RatecodeID string, "
        "PULocationID string, DOLocationID string, airport_column string"
    ),
)
schema_summary.orderBy("month").show(12, truncate=False)


+-------+---------------+----------+------------+------------+--------------+
|month  |passenger_count|RatecodeID|PULocationID|DOLocationID|airport_column|
+-------+---------------+----------+------------+------------+--------------+
|2023-01|double         |double    |bigint      |bigint      |airport_fee   |
|2023-02|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-03|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-04|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-05|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-06|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-07|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-08|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-09|bigint         |bigint    |int         |int         |Airport_fee   |
|2023-10|bigint         |bigint    |int         |int         |Ai

### Confirm the complete January schema
The timestamp types and original numerical columns below will guide consistent loading and target construction. No inference about data quality or model performance is made from schema types alone.


In [5]:
january_df = spark.read.parquet(str(MONTHLY_FILES[0]))
print("Source:", MONTHLY_FILES[0].name)
january_df.printSchema()


Source: yellow_tripdata_2023-01.parquet
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



## 4. Standardise and combine the monthly files

Each monthly Parquet file is read separately using its source schema,
then converted to consistent column names and types.

Identifier columns use long integers. Passenger counts and rate codes
remain doubles at this stage so their original values are available
for data quality checks. Distance and monetary columns use doubles.

The airport fee column is consistently named airport_fee. Pickup and
drop-off timestamps are interpreted using America/New_York and cast
to Spark timestamp types, following the timezone convention in
Assessment 1.

The standardised monthly DataFrames are combined using unionByName.
This loading stage retains all source records.

In [6]:
from functools import reduce
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/New_York")

COLUMN_TYPES = {
    "VendorID": "long",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",
    "trip_distance": "double",
    "RatecodeID": "double",
    "store_and_fwd_flag": "string",
    "PULocationID": "long",
    "DOLocationID": "long",
    "payment_type": "long",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
}


def standardise_month(source_path):
    source_df = spark.read.parquet(str(source_path))

    # Resolve source names, including Airport_fee versus airport_fee.
    source_columns = {
        name.lower(): name for name in source_df.columns
    }

    missing_columns = [
        name for name in COLUMN_TYPES
        if name.lower() not in source_columns
    ]
    if missing_columns:
        raise ValueError(
            f"{source_path.name}: missing columns {missing_columns}"
        )

    return source_df.select(
        *[
            F.col(source_columns[name.lower()])
            .cast(data_type)
            .alias(name)
            for name, data_type in COLUMN_TYPES.items()
        ]
    )


monthly_dfs = [
    standardise_month(source_path)
    for source_path in MONTHLY_FILES
]

raw_trips_df = reduce(
    lambda combined, month: combined.unionByName(month),
    monthly_dfs,
)

raw_trips_df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [7]:
raw_trip_count = raw_trips_df.count()

print("Monthly files:", len(monthly_dfs))
print("Columns:", len(raw_trips_df.columns))
print(f"Total trip records: {raw_trip_count:,}")

raw_trips_df.select(
    "tpep_pickup_datetime",
    "passenger_count",
    "RatecodeID",
    "PULocationID",
    "airport_fee",
).show(5, truncate=False)

Monthly files: 12
Columns: 19
Total trip records: 38,310,226
+--------------------+---------------+----------+------------+-----------+
|tpep_pickup_datetime|passenger_count|RatecodeID|PULocationID|airport_fee|
+--------------------+---------------+----------+------------+-----------+
|2023-01-01 00:32:10 |1.0            |1.0       |161         |0.0        |
|2023-01-01 00:55:08 |1.0            |1.0       |43          |0.0        |
|2023-01-01 00:25:04 |1.0            |1.0       |48          |0.0        |
|2023-01-01 00:03:48 |0.0            |1.0       |138         |1.25       |
|2023-01-01 00:10:29 |1.0            |1.0       |107         |0.0        |
+--------------------+---------------+----------+------------+-----------+
only showing top 5 rows

